# 05 — Bias and fairness audit

Compute segment-level demographic parity, equal opportunity (TPR parity), and FPR parity. Writes `mlruns/fairness_snapshot.json` consumed by the Streamlit fairness-audit page. Required by US SR 11-7 model risk management.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.loader import LABEL_COLUMN, PAYMENT_FORMAT_COLUMN, DataLoader
from src.models.ensemble import AMLEnsemble
from src.monitoring.fairness import generate_fairness_snapshot

sns.set_theme(style='whitegrid', palette='deep')

In [ ]:
loader = DataLoader()
frame = loader.load_sample(n=200_000, random_state=42)
ensemble = AMLEnsemble.load(Path('models/ensemble.pkl'))
frame['prediction'] = ensemble.predict(frame)
print(f'Alert rate (global): {frame["prediction"].mean():.4f}')

## 1. Segment-level audit on payment format

In [ ]:
snapshot = generate_fairness_snapshot(
    frame=frame,
    segment_column=PAYMENT_FORMAT_COLUMN,
    label_column=LABEL_COLUMN,
    prediction_column='prediction',
)
segments = pd.DataFrame(snapshot['segments'])
segments[['segment', 'n', 'alert_rate', 'true_positive_rate', 'false_positive_rate']]

## 2. Parity gaps

In [ ]:
gaps = pd.DataFrame(snapshot['parity_gaps'])
print(gaps[['metric', 'gap', 'severity', 'min_segment', 'max_segment']])

## 3. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ['alert_rate', 'true_positive_rate', 'false_positive_rate']):
    ax.bar(segments['segment'], segments[metric], color='#0f172a')
    ax.set_title(metric.replace('_', ' ').title())
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()